In [13]:
from langchain_community.document_loaders import TextLoader # raw book text -> LangChain format
from langchain_text_splitters import CharacterTextSplitter # split text into chunks
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings  # generate embeddings
from langchain_chroma import Chroma # vector store
import os

In [14]:
from dotenv import load_dotenv

load_dotenv()

True

In [15]:
import pandas as pd

books = pd.read_csv('books_cleaned.csv')

In [16]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [17]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [18]:
books["tagged_description"].to_csv("tagged_descriptions.txt",
                                   sep = "\n",
                                   index = False,
                                   header = False)

In [20]:
# raw_documents = TextLoader("tagged_descriptions.txt", encoding="utf-8").load()
# text_splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator="\n")  # split on newlines only
# documents = text_splitter.split_documents(raw_documents)




from langchain_core.documents import Document

# Load the file
with open("tagged_descriptions.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Split on newlines manually
lines = text.split("\n")

# Create a Document for each line
documents = [Document(page_content=line) for line in lines if line.strip()]  # skip empty lines

In [21]:
documents[0]

Document(metadata={}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and 

In [22]:
import os
from typing import List
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [23]:
# 1. Custom embedding class (Completely free, no API limits)
class LocalSentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # BGE convention: documents/passages get encoded plain, no instruction prefix
        return self.model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text: str) -> List[float]:
        # BGE convention: only the QUERY gets an instruction prefix, this is the one
        # required change from MiniLL and it's the main thing that improves relevance
        instructed = f"Represent this sentence for searching relevant passages: {text}"
        return self.model.encode(instructed).tolist()



In [25]:
# Initialize the retrieval-tuned encoder (still ~130MB, still fully local/free)
embeddings = LocalSentenceTransformerEmbeddings("BAAI/bge-small-en-v1.5")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [26]:
# # Running this cell results in tedious training, so skip to the next call if you have a saved vector store / ran this one already for the first time

# Split your long documents so they fit the model's window

# Modify the chunking step to include ISBN13 in metadata
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Add ISBN13 to metadata before chunking
documents_with_metadata = []
for doc in documents:
    # Extract ISBN13 from the beginning of the content
    isbn13 = doc.page_content.split()[0].strip('"')
    # Store ISBN13 both in metadata and at the start of content
    documents_with_metadata.append(
        Document(page_content=f"{isbn13} {doc.page_content}", metadata={"isbn13": isbn13})
    )

# Now chunk the documents with metadata
final_chunks = text_splitter.split_documents(documents_with_metadata)

# Create vector store with persistence
db_books = Chroma.from_documents(
    final_chunks,
    embeddings,
    collection_name="books",
    persist_directory="./chroma_db"  # Save to disk
)

print(f"Successfully embedded {len(final_chunks)} text chunks with zero rate limits!")
print(f"Vector store saved to ./chroma_db")

KeyboardInterrupt: 

In [27]:
# Load vector store from disk if it exists (skip training next time)
import os
from langchain_chroma import Chroma

if os.path.exists("./chroma_db"):
    print("Loading existing vector store from disk...")
    db_books = Chroma(
        collection_name="books",
        embedding_function=embeddings,
        persist_directory="./chroma_db"
    )
    print("Vector store loaded successfully!")
else:
    print("No saved vector store found. Run cell 9 to create it.")

Loading existing vector store from disk...
Vector store loaded successfully!


In [28]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='5aa0ed25-68b0-437a-ae92-6d8f0b39e418', metadata={'isbn13': '9780786808069'}, page_content='9780786808069 9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.'),
 Document(id='311bbb78-58d1-4569-9652-112b0755af52', metadata={'isbn13': '9780786808069'}, page_content='9780786808069 9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.'),
 Document(id='2d3ea776-eace-4738-85fa-ef3ab77c771c', metadata={'isbn13': '9780374522599'}, page_content="9780374522599 9780374522599 The Control of Nature is John McPhee's bestselling account of place

In [29]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [13]:
'''def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)
    print(f"Got {len(recs)} results")

    # Extract ISBN13 from metadata
    books_list = []
    for rec in recs:
        if "isbn13" in rec.metadata:
            books_list.append(rec.metadata["isbn13"])

    print(f"Extracted {len(books_list)} ISBN13s: {books_list[:5]}")

    # Remove duplicates while preserving order
    # seen = set()
    # unique_books = []
    # for isbn in books_list:
    #     if isbn not in seen:
    #         seen.add(isbn)
    #         unique_books.append(isbn)

    # Convert to strings for comparison
    books_list_str = [str(isbn) for isbn in books_list]

    return books[books["isbn13"].astype(str).isin(books_list_str)].head(top_k)'''

'def retrieve_semantic_recommendations(\n        query: str,\n        top_k: int = 10,\n) -> pd.DataFrame:\n    recs = db_books.similarity_search(query, k=50)\n    print(f"Got {len(recs)} results")\n\n    # Extract ISBN13 from metadata\n    books_list = []\n    for rec in recs:\n        if "isbn13" in rec.metadata:\n            books_list.append(rec.metadata["isbn13"])\n\n    print(f"Extracted {len(books_list)} ISBN13s: {books_list[:5]}")\n\n    # Remove duplicates while preserving order\n    # seen = set()\n    # unique_books = []\n    # for isbn in books_list:\n    #     if isbn not in seen:\n    #         seen.add(isbn)\n    #         unique_books.append(isbn)\n\n    # Convert to strings for comparison\n    books_list_str = [str(isbn) for isbn in books_list]\n\n    return books[books["isbn13"].astype(str).isin(books_list_str)].head(top_k)'

In [14]:
'''# Check if results have metadata
recs = db_books.similarity_search("A book to teach children about nature", k=5)
for i, rec in enumerate(recs):
    print(f"Result {i}: metadata = {rec.metadata}")
    print(f"  Content starts with: {rec.page_content[:80]}")'''

'# Check if results have metadata\nrecs = db_books.similarity_search("A book to teach children about nature", k=5)\nfor i, rec in enumerate(recs):\n    print(f"Result {i}: metadata = {rec.metadata}")\n    print(f"  Content starts with: {rec.page_content[:80]}")'

In [15]:
'''retrieve_semantic_recommendations("A book to teach children about nature")'''

'retrieve_semantic_recommendations("A book to teach children about nature")'

In [30]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)
    print(f"Got {len(recs)} results")

    books_list = []
    for rec in recs:
        if "isbn13" in rec.metadata:
            books_list.append(rec.metadata["isbn13"])

    print(f"Extracted {len(books_list)} ISBN13s: {books_list[:5]}.")

    books_list_str = [str(isbn) for isbn in books_list]

    result = books[books["isbn13"].astype(str).isin(books_list_str)].head(top_k)
    print(f"Result: {len(result)} rows")
    return result

In [31]:
retrieve_semantic_recommendations("A book to teach children about nature")

Got 50 results
Extracted 50 ISBN13s: ['9780786808069', '9780786808069', '9780374522599', '9780374522599', '9780143037392'].
Result: 10 rows


,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
31,9780007105045,0007105045,Tree and Leaf,John Ronald Reuel Tolkien,Literary Collections,http://books.google.com/books/content?id=aPb_A...,"""The two works 'On fairy-stories' and 'Leaf by...",2001.0,4.09,176.0,2245.0,Tree and Leaf: The Homecoming of Beorhtnoth : ...,"9780007105045 ""The two works 'On fairy-stories..."
89,9780060005696,0060005696,The Paradox of Choice,Barry Schwartz,Business & Economics,NaN,The author of The Battle for Human Nature expl...,2005.0,3.84,265.0,23734.0,The Paradox of Choice: Why More Is Less,9780060005696 The author of The Battle for Hum...
324,9780060959036,0060959037,Prodigal Summer,Barbara Kingsolver,Fiction,http://books.google.com/books/content?id=06IwG...,Barbara Kingsolver's fifth novel is a hymn to ...,2001.0,4.00,444.0,85440.0,Prodigal Summer: A Novel,9780060959036 Barbara Kingsolver's fifth novel...
373,9780061131622,0061131628,Mandy,Julie Andrews Edwards,Juvenile Fiction,http://books.google.com/books/content?id=5dNj_...,"Mandy, a ten-year-old orphan, dreams of a plac...",2006.0,4.24,320.0,9482.0,Mandy,"9780061131622 Mandy, a ten-year-old orphan, dr..."
442,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5S...,"First published more than three decades ago, t...",1998.0,4.39,112.0,1160.0,The Sense of Wonder,9780067575208 First published more than three ...
692,9780140448009,0140448004,Three Tales,Gustave Flaubert;Roger Whitehouse;Geoffrey Wall,Fiction,http://books.google.com/books/content?id=XFzga...,Features short fiction by the French naturalis...,2005.0,3.71,110.0,3050.0,Three Tales,9780140448009 Features short fiction by the Fr...
799,9780142003343,0142003344,The Blank Slate,Steven Pinker,Psychology,http://books.google.com/books/content?id=7rJ5g...,In a study of the nature versus nurture debate...,2003.0,4.08,528.0,17851.0,The Blank Slate: The Modern Denial of Human Na...,9780142003343 In a study of the nature versus ...
855,9780143037392,0143037390,The Read-aloud Handbook,Jim Trelease,Language Arts & Disciplines,http://books.google.com/books/content?id=B2_yU...,Explains the importance of reading aloud to ch...,2006.0,4.40,432.0,4122.0,The Read-aloud Handbook,9780143037392 Explains the importance of readi...
997,9780195108965,0195108965,Notebooks of the Mind,Vera John-Steiner,Psychology,http://books.google.com/books/content?id=uj1RD...,How do creative people think? Do great works o...,1997.0,3.81,288.0,17.0,Notebooks of the Mind: Explorations of Thinking,9780195108965 How do creative people think? Do...
1639,9780374422080,0374422087,Everything on a Waffle,Polly Horvath,Juvenile Fiction,http://books.google.com/books/content?id=NimVJ...,This Newbery Honor Book tells the story of 11 ...,2004.0,3.71,150.0,9631.0,Everything on a Waffle,9780374422080 This Newbery Honor Book tells th...


In [32]:
# Test if similarity search works
recs = db_books.similarity_search("A book to teach children about nature", k=5)
print(f"Number of results: {len(recs)}")
if len(recs) > 0:
    print(f"First result content: {recs[0].page_content[:100]}")
    print(f"First result metadata: {recs[0].metadata}")

Number of results: 5
First result content: 9780786808069 9780786808069 Children will discover the exciting world of their own backyard in this 
First result metadata: {'isbn13': '9780786808069'}


In [33]:
# Check what ISBN13 values are in metadata vs books dataframe
recs = db_books.similarity_search("A book to teach children about nature", k=5)
books_list = []
for rec in recs:
    if "isbn13" in rec.metadata:
        books_list.append(rec.metadata["isbn13"])
        print(f"Metadata ISBN13: {rec.metadata['isbn13']}")

print(f"\nBooks list: {books_list}")
print(f"Sample ISBN13 from books df: {books['isbn13'].head()}")

Metadata ISBN13: 9780786808069
Metadata ISBN13: 9780786808069
Metadata ISBN13: 9780374522599
Metadata ISBN13: 9780374522599
Metadata ISBN13: 9780143037392

Books list: ['9780786808069', '9780786808069', '9780374522599', '9780374522599', '9780143037392']
Sample ISBN13 from books df: 0    9780002005883
1    9780002261982
2    9780006178736
3    9780006280897
4    9780006280934
Name: isbn13, dtype: int64


In [34]:
# Check if chunks have metadata
print(f"First chunk metadata: {final_chunks[0].metadata}")
print(f"First chunk content: {final_chunks[0].page_content[:100]}")

First chunk metadata: {'isbn13': '9780002005883'}
First chunk content: 9780002005883 9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over


In [35]:
retrieve_semantic_recommendations("A book to teach children about nature")

Got 50 results
Extracted 50 ISBN13s: ['9780786808069', '9780786808069', '9780374522599', '9780374522599', '9780143037392'].
Result: 10 rows


,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
31,9780007105045,0007105045,Tree and Leaf,John Ronald Reuel Tolkien,Literary Collections,http://books.google.com/books/content?id=aPb_A...,"""The two works 'On fairy-stories' and 'Leaf by...",2001.0,4.09,176.0,2245.0,Tree and Leaf: The Homecoming of Beorhtnoth : ...,"9780007105045 ""The two works 'On fairy-stories..."
89,9780060005696,0060005696,The Paradox of Choice,Barry Schwartz,Business & Economics,NaN,The author of The Battle for Human Nature expl...,2005.0,3.84,265.0,23734.0,The Paradox of Choice: Why More Is Less,9780060005696 The author of The Battle for Hum...
324,9780060959036,0060959037,Prodigal Summer,Barbara Kingsolver,Fiction,http://books.google.com/books/content?id=06IwG...,Barbara Kingsolver's fifth novel is a hymn to ...,2001.0,4.00,444.0,85440.0,Prodigal Summer: A Novel,9780060959036 Barbara Kingsolver's fifth novel...
373,9780061131622,0061131628,Mandy,Julie Andrews Edwards,Juvenile Fiction,http://books.google.com/books/content?id=5dNj_...,"Mandy, a ten-year-old orphan, dreams of a plac...",2006.0,4.24,320.0,9482.0,Mandy,"9780061131622 Mandy, a ten-year-old orphan, dr..."
442,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5S...,"First published more than three decades ago, t...",1998.0,4.39,112.0,1160.0,The Sense of Wonder,9780067575208 First published more than three ...
692,9780140448009,0140448004,Three Tales,Gustave Flaubert;Roger Whitehouse;Geoffrey Wall,Fiction,http://books.google.com/books/content?id=XFzga...,Features short fiction by the French naturalis...,2005.0,3.71,110.0,3050.0,Three Tales,9780140448009 Features short fiction by the Fr...
799,9780142003343,0142003344,The Blank Slate,Steven Pinker,Psychology,http://books.google.com/books/content?id=7rJ5g...,In a study of the nature versus nurture debate...,2003.0,4.08,528.0,17851.0,The Blank Slate: The Modern Denial of Human Na...,9780142003343 In a study of the nature versus ...
855,9780143037392,0143037390,The Read-aloud Handbook,Jim Trelease,Language Arts & Disciplines,http://books.google.com/books/content?id=B2_yU...,Explains the importance of reading aloud to ch...,2006.0,4.40,432.0,4122.0,The Read-aloud Handbook,9780143037392 Explains the importance of readi...
997,9780195108965,0195108965,Notebooks of the Mind,Vera John-Steiner,Psychology,http://books.google.com/books/content?id=uj1RD...,How do creative people think? Do great works o...,1997.0,3.81,288.0,17.0,Notebooks of the Mind: Explorations of Thinking,9780195108965 How do creative people think? Do...
1639,9780374422080,0374422087,Everything on a Waffle,Polly Horvath,Juvenile Fiction,http://books.google.com/books/content?id=NimVJ...,This Newbery Honor Book tells the story of 11 ...,2004.0,3.71,150.0,9631.0,Everything on a Waffle,9780374422080 This Newbery Honor Book tells th...
